# 
RLCT Estimation of Multitask Sparse Parity

In [1]:
%pip install devinterp seaborn torchvision pickleshare wandb plotly einops scikit-learn
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD
%cd ..

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.10.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.6 MB)
  Using cached cloudpickle-3.1.0-py3-none-any.whl (22 kB)

[notice] A new release of pip is available: 23.1.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'entropy-sgd' already exists and is not an empty directory.
/gpfs/home1/bshaffrey/entropy-sgd
/gpfs/home1/bshaffrey


In [2]:
import numpy as np
import torch as t
import torch
import torch.nn as nn
import torch.optim as optim
import time
import torch.nn.functional as F
import einops
import random
from dataclasses import dataclass
import os
import copy
import wandb
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from python.optim import EntropySGD
from torch.utils.data import DataLoader
from collections import defaultdict
from itertools import islice, product
import random
import time
import math
from pathlib import Path

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT

PRIMARY, SECONDARY, TERTIARY, QUATERNARY, QUINARY, SENARY = sns.color_palette("muted")[:6]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT, QUINARY_LIGHT, SENARY_LIGHT = sns.color_palette(
    "pastel"
)[:6]

print(len(sns.color_palette("pastel")))

10


In [13]:
model = 'mlp'
models_saved = torch.load('models_saved_2000_mlp.pt')
info = torch.load('info_2000_mlp.pt')
train_loader = torch.load('train_loader_2000_mlp.pt')
train_loaders_subtasks = torch.load('train_loaders_subtasks_2000_mlp.pt')
rlct_estimates_final = torch.load('rlct_estimates_full_' + str(2000) + '_' + model + '.pt')
loss_diffs_per_task = torch.load('loss_diffs_per_task_' + str(2000) + '_' + model + '.pt')

/scratch-local/bshaffrey.9108048/ipykernel_621847/2525625568.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  models_saved = torch.load('models_saved_2000_mlp.pt')
/scrat

In [3]:
class FastTensorDataLoader:
    """
    A DataLoader-like object for a set of tensors that can be much faster than
    TensorDataset + DataLoader because dataloader grabs individual indices of
    the dataset and calls cat (slow).
    """
    def __init__(self, *tensors, batch_size=32, shuffle=False):
        """
        Initialize a FastTensorDataLoader.

        :param *tensors: tensors to store. Must have the same length @ dim 0.
        :param batch_size: batch size to load.
        :param shuffle: if True, shuffle the data *in-place* whenever an
            iterator is created out of this object.

        :returns: A FastTensorDataLoader.
        """
        assert all(t.shape[0] == tensors[0].shape[0] for t in tensors)
        self.tensors = tensors

        self.dataset_len = self.tensors[0].shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Calculate # batches
        n_batches, remainder = divmod(self.dataset_len, self.batch_size)
        if remainder > 0:
            n_batches += 1
        self.n_batches = n_batches

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=self.tensors[0].device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        if self.indices is not None:
            indices = self.indices[self.i:self.i+self.batch_size]
            batch = tuple(torch.index_select(t, 0, indices) for t in self.tensors)
        else:
            batch = tuple(t[self.i:self.i+self.batch_size] for t in self.tensors)
        self.i += self.batch_size
        return batch

    def __len__(self):
        return self.n_batches


def get_batch(n_tasks, n, Ss, codes, sizes, device='cpu', dtype=torch.float32):
    """Creates batch. 

    Parameters
    ----------
    n_tasks : int
        Number of tasks.
    n : int
        Bit string length for sparse parity problem.
    Ss : list of lists of ints
        Subsets of [1, ... n] to compute sparse parities on.
    codes : list of int
        The subtask indices which the batch will consist of
    sizes : list of int
        Number of samples for each subtask
    device : str
        Device to put batch on.
    dtype : torch.dtype
        Data type to use for input x. Output y is torch.int64.

    Returns
    -------
    x : torch.Tensor
        inputs
    y : torch.Tensor
        labels
    """
    batch_x = torch.zeros((sum(sizes), n_tasks+n), dtype=dtype, device=device)
    batch_y = torch.zeros((sum(sizes),), dtype=torch.int64, device=device)
    start_i = 0
    for (S, size, code) in zip(Ss, sizes, codes):
        if size > 0:
            x = torch.randint(low=0, high=2, size=(size, n), dtype=dtype, device=device)
            y = torch.sum(x[:, S], dim=1) % 2
            x_task_code = torch.zeros((size, n_tasks), dtype=dtype, device=device)
            x_task_code[:, code] = 1
            x = torch.cat([x_task_code, x], dim=1)
            batch_x[start_i:start_i+size, :] = x
            batch_y[start_i:start_i+size] = y
            start_i += size
    return batch_x, batch_y
    
def cycle(iterable):
    while True:
        for x in iterable:
            yield x


In [4]:
class MLP(nn.Module):
    
    def __init__(self, activation, depth, width):
        super(MLP, self).__init__()
        
        if activation == 'ReLU':
            activation_fn = nn.ReLU
        elif activation == 'Tanh':
            activation_fn = nn.Tanh
        elif activation == 'Sigmoid':
            activation_fn = nn.Sigmoid
        else:
            assert False, f"Unrecognized activation function identifier: {activation}"

        # create model
        layers = []
        for i in range(depth):
            if i == 0:
                layers.append(nn.Linear(n_tasks + n, width))
                layers.append(activation_fn())
            elif i == depth - 1:
                layers.append(nn.Linear(width, 2))
            else:
                layers.append(nn.Linear(width, width))
                layers.append(activation_fn())
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

In [5]:
seed = 1
np.random.seed(seed)
torch.manual_seed(seed)

torch.set_default_tensor_type(torch.DoubleTensor)


class BioLinear(nn.Module):

    def __init__(self, in_dim, out_dim, in_fold=1, out_fold=1):
        super(BioLinear, self).__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.linear = nn.Linear(in_dim, out_dim)
        self.in_fold = in_fold
        self.out_fold = out_fold
        assert in_dim % in_fold == 0
        assert out_dim % out_fold == 0
        #compute in_cor, shape: (in_dim)
        in_dim_fold = int(in_dim/in_fold)
        out_dim_fold = int(out_dim/out_fold)
        self.in_coordinates = torch.tensor(list(np.linspace(1/(2*in_dim_fold), 1-1/(2*in_dim_fold), num=in_dim_fold))*in_fold, dtype=torch.float)
        self.out_coordinates = torch.tensor(list(np.linspace(1/(2*out_dim_fold), 1-1/(2*out_dim_fold), num=out_dim_fold))*out_fold, dtype=torch.float)
        
    def forward(self, x):
        return self.linear(x)

/home/bshaffrey/.local/lib/python3.11/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


In [6]:
class BioMLP(nn.Module):
    def __init__(self, in_dim=2, out_dim=2, w=2, depth=2, shp=None, token_embedding=False, embedding_size=None):
        super(BioMLP, self).__init__()
        if shp == None:
            shp = [in_dim] + [w]*(depth-1) + [out_dim]
            self.in_dim = in_dim
            self.out_dim = out_dim
            self.depth = depth
                 
        else:
            self.in_dim = shp[0]
            self.out_dim = shp[-1]
            self.depth = len(shp) - 1
        linear_list = []
        for i in range(self.depth):
            if i == 0:
                # for modular addition
                #linear_list.append(BioLinear(shp[i], shp[i+1], in_fold=2))
                # for regression
                linear_list.append(BioLinear(shp[i], shp[i+1], in_fold=1))
            else:
                linear_list.append(BioLinear(shp[i], shp[i+1]))
        self.linears = nn.ModuleList(linear_list)
        
        
        if token_embedding == True:
            # embedding size: number of tokens * embedding dimension
            self.embedding = torch.nn.Parameter(torch.normal(0,1,size=embedding_size))
        
        self.shp = shp
        # parameters for the bio-inspired trick
        self.l0 = 0.5 # distance between two nearby layers
        self.in_perm = nn.Parameter(torch.tensor(np.arange(int(self.in_dim/self.linears[0].in_fold)), dtype=torch.float))
        self.out_perm = nn.Parameter(torch.tensor(np.arange(int(self.out_dim/self.linears[-1].out_fold)), dtype=torch.float))
        self.top_k = 30
        self.token_embedding = token_embedding

    def forward(self, x):
        shp = x.shape
        in_fold = self.linears[0].in_fold
        x = x.reshape(shp[0], in_fold, int(shp[1]/in_fold))
        x = x[:,:,self.in_perm.long()]
        x = x.reshape(shp[0], shp[1])
        f = torch.nn.SiLU()
        for i in range(self.depth-1):
            x = f(self.linears[i](x))
        x = self.linears[-1](x)
        
        out_perm_inv = torch.zeros(self.out_dim, dtype=torch.long)
        out_perm_inv[self.out_perm.long()] = torch.arange(self.out_dim)
        x = x[:,out_perm_inv]
        
        return x
    
    def get_linear_layers(self):
        return self.linears
    
    def get_cc(self, weight_factor=2.0, bias_penalize=True, no_penalize_last=False):
        # compute connection cost
        cc = 0
        num_linear = len(self.linears)
        for i in range(num_linear):
            if i == num_linear - 1 and no_penalize_last:
                weight_factor = 0.
            biolinear = self.linears[i].to('cuda')
            dist = torch.abs(biolinear.out_coordinates.unsqueeze(dim=1) - biolinear.in_coordinates.unsqueeze(dim=0)).to('cuda')
            cc += torch.mean(torch.abs(biolinear.linear.weight)*(weight_factor*dist+self.l0))
            if bias_penalize == True:
                cc += torch.mean(torch.abs(biolinear.linear.bias)*(self.l0))
        if self.token_embedding:
            cc += torch.mean(torch.abs(self.embedding)*(self.l0))
        return cc
    
    def swap_weight(self, weights, j, k, swap_type="out"):
        with torch.no_grad():  
            if swap_type == "in":
                temp = weights[:,j].clone()
                weights[:,j] = weights[:,k].clone()
                weights[:,k] = temp
            elif swap_type == "out":
                temp = weights[j].clone()
                weights[j] = weights[k].clone()
                weights[k] = temp
            else:
                raise Exception("Swap type {} is not recognized!".format(swap_type))
            
    def swap_bias(self, biases, j, k):
        with torch.no_grad():  
            temp = biases[j].clone()
            biases[j] = biases[k].clone()
            biases[k] = temp
    
    def swap(self, i, j, k):
        # in the ith layer (of neurons), swap the jth and the kth neuron. 
        # Note: n layers of weights means n+1 layers of neurons.
        # (incoming, outgoing) * weights + biases are swapped. 
        linears = self.get_linear_layers()
        num_linear = len(linears)
        if i == 0:
            # input layer, only has outgoing weights; update in_perm
            weights = linears[i].linear.weight
            infold = linears[i].in_fold
            fold_dim = int(weights.shape[1]/infold)
            for l in range(infold):
                self.swap_weight(weights, j+fold_dim*l, k+fold_dim*l, swap_type="in")
            # change input_perm
            self.swap_bias(self.in_perm, j, k)
        elif i == num_linear:
            # output layer, only has incoming weights and biases; update out_perm
            weights = linears[i-1].linear.weight
            biases = linears[i-1].linear.bias
            self.swap_weight(weights, j, k, swap_type="out")
            self.swap_bias(biases, j, k)
            # change output_perm
            self.swap_bias(self.out_perm, j, k)
        else:
            # middle layer : (incoming, outgoing) * weights, and biases
            weights_in = linears[i-1].linear.weight
            weights_out = linears[i].linear.weight
            biases = linears[i-1].linear.bias
            self.swap_weight(weights_in, j, k, swap_type="out")
            self.swap_weight(weights_out, j, k, swap_type="in")
            self.swap_bias(biases, j, k)

    def get_top_id(self, i, top_k=20):
        linears = self.get_linear_layers()
        num_linear = len(linears)
        if i == 0:
            # input layer
            weights = linears[i].linear.weight
            score = torch.sum(torch.abs(weights), dim=0)
            in_fold = linears[0].in_fold
            score = torch.sum(score.reshape(in_fold, int(score.shape[0]/in_fold)), dim=0)
        elif i == num_linear:
            # output layer
            weights = linears[i-1].linear.weight
            score = torch.sum(torch.abs(weights), dim=1)
        else:
            weights_in = linears[i-1].linear.weight
            weights_out = linears[i].linear.weight
            score = torch.sum(torch.abs(weights_out), dim=0) + torch.sum(torch.abs(weights_in), dim=1)
        top_index = torch.flip(torch.argsort(score),[0])[:top_k]
        return top_index
    
    def relocate_ij(self, i, j):
        # In the ith layer (of neurons), relocate the jth neuron
        linears = self.get_linear_layers()
        num_linear = len(linears)
        if i < num_linear:
            num_neuron = int(linears[i].linear.weight.shape[1]/linears[i].in_fold)
        else:
            num_neuron = linears[i-1].linear.weight.shape[0]
        ccs = []
        for k in range(num_neuron):
            self.swap(i,j,k)
            cc = self.get_cc()
            ccs.append(cc)
            self.swap(i,j,k)
        k = torch.argmin(torch.stack(ccs))
        self.swap(i,j,k)
            
    def relocate_i(self, i):
        # Relocate neurons in the ith layer
        top_id = self.get_top_id(i, top_k=self.top_k)
        for j in top_id:
            self.relocate_ij(i,j)
            
    def relocate(self):
        # Relocate neurons in the whole model
        linears = self.get_linear_layers()
        num_linear = len(linears)
        for i in tqdm(range(num_linear+1)):
            self.relocate_i(i)
            
    def plot(self):
        fig, ax = plt.subplots(figsize=(6,6))
        shp = self.shp
        s = 1/(2*max(shp))
        for j in range(len(shp)):
            N = shp[j]
            if j == 0:
                in_fold = self.linears[j].in_fold
                N = int(N/in_fold)
            for i in range(N):
                if j == 0:
                    for fold in range(in_fold):
                        circle = Ellipse((1/(2*N)+i/N, 0.1*j+0.02*fold-0.01), s, s/10*((len(shp)-1)+0.4), color='black')
                        ax.add_patch(circle)
                else:
                    for fold in range(in_fold):
                        circle = Ellipse((1/(2*N)+i/N, 0.1*j), s, s/10*((len(shp)-1)+0.4), color='black')
                        ax.add_patch(circle)


        plt.ylim(-0.02,0.1*(len(shp)-1)+0.02)
        plt.xlim(-0.02,1.02)

        linears = self.linears
        for ii in range(len(linears)):
            biolinear = linears[ii]
            p = biolinear.linear.weight.clone()
            p_shp = p.shape
            
            p = p/torch.abs(p).max()
            in_fold = biolinear.in_fold
            fold_num = int(p_shp[1]/in_fold)
            for i in range(p_shp[0]):
                if ii == 0:
                    for fold in range(in_fold):
                        for j in range(fold_num):
                            plt.plot([1/(2*p_shp[0])+i/p_shp[0], 1/(2*fold_num)+j/fold_num], [0.1*(ii+1),0.1*ii+0.02*fold-0.01], lw=1*np.abs(p[i,j].detach().numpy()), color="blue" if p[i,j]>0 else "red")
                else:
                    for j in range(fold_num):
                        plt.plot([1/(2*p_shp[0])+i/p_shp[0], 1/(2*fold_num)+j/fold_num], [0.1*(ii+1),0.1*ii], lw=0.5*np.abs(p[i,j].detach().numpy()), color="blue" if p[i,j]>0 else "red")
                    
        ax.axis('off')

In [7]:
def get_subsets(n_tasks, n, k):
    Ss = []
    for _ in range(n_tasks * 10):
        S = tuple(sorted(list(random.sample(range(n), k))))
        if S not in Ss:
            Ss.append(S)
        if len(Ss) == n_tasks:
            break
    assert len(Ss) == n_tasks, "Couldn't find enough subsets for tasks for the given n, k"
    return Ss

def get_data(steps, batch_size, cdf, n_tasks, n, Ss, device, dtype, test_points_per_task):
    x, y = torch.zeros((steps * batch_size, n_tasks + n), dtype=dtype), torch.zeros((steps * batch_size), dtype=torch.int64)
    x_sub, y_sub = list(), list()
    for task in range(n_tasks):
        x_sub.append(torch.zeros((steps * test_points_per_task, n_tasks + n), dtype=dtype))
        y_sub.append(torch.zeros((steps * test_points_per_task), dtype=torch.int64))
    for step in tqdm(range(steps)):
        samples = np.searchsorted(cdf, np.random.rand(batch_size,))
        hist, _ = np.histogram(samples, bins=n_tasks, range=(0, n_tasks-1))
        x[batch_size * step : batch_size * (step + 1), : ], y[batch_size * step : batch_size * (step + 1)] = get_batch(n_tasks=n_tasks, n=n, Ss=Ss, codes=list(range(n_tasks)), sizes=hist, device=device, dtype=dtype)
        for task in range(n_tasks):
           x_sub[task][test_points_per_task * step : test_points_per_task * (step + 1), : ], y_sub[task][test_points_per_task * step : test_points_per_task * (step + 1)] = get_batch(n_tasks=n_tasks, n=n, Ss=[Ss[task]], codes=[task], sizes=[test_points_per_task], device=device, dtype=dtype)
    train_data = list(zip(x, y))
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    train_loaders_subtasks = []

    for task in range(n_tasks):
        train_data_sub = list(zip(x_sub[task], y_sub[task]))
        train_loaders_subtasks.append(torch.utils.data.DataLoader(train_data_sub, batch_size=batch_size, shuffle=True))
    return train_loader, train_loaders_subtasks

In [8]:
def accuracy_function(outputs, targets):
    return (outputs.argmax(1) == targets).float().mean()
    #torch.sum(labels_i_pred == y_i).item() / test_points

def train_one_epoch(model, train_loader, train_loaders_subtasks, optimizer, criterion, device, n_tasks, lamb):
    model.train()
    train_loss = 0
    train_accuracy = 0
    
    for (data, targets) in train_loader:
        optimizer.zero_grad()
        outputs = model(data.to(device))
        loss = criterion(outputs, targets.to(device))
        train_loss += loss.item()
        train_accuracy += accuracy_function(outputs, targets.to(device)).item()
        #cc = model.get_cc(weight_factor=2.0, no_penalize_last=False).to(device)
        cc = 0.0
        #regularisation_term = l1_loss_zero_one(model.parameters())
        regularisation_term = neuron_level_regularisation(model)
        total_loss = loss + lamb * regularisation_term
        total_loss.backward()
        optimizer.step()
        
    return train_loss / len(train_loader), train_accuracy / len(train_loader)


def evaluate(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    test_accuracy = 0
    with torch.no_grad():
        for index, (data, targets) in enumerate(test_loader):
            outputs = model(data.to(device))
            loss = criterion(outputs, targets.to(device))
            test_loss += loss.item()
            test_accuracy += accuracy_function(outputs, targets.to(device)).item()
            
    return test_loss / len(test_loader), test_accuracy / len(test_loader)


In [ ]:
def l0_regulariser(parameters, alpha=1.0):
    loss = 0
    for param in parameters:
        loss += torch.sum(param != 0).float()
    return alpha * loss

def l1_loss_zero_one(parameters):
    l1_loss = 0
    for param in parameters:
        l1_loss += torch.min(param.abs(), (param - 1).abs()).sum()
    return l1_loss

def double_well_regulariser(parameters, beta=1.0):
    loss = 0
    for param in parameters:
        loss += beta * (((param - .5)** 2 - .25) ** 2).sum()
    return loss

def neuron_level_regularisation(model):
    penalty = 0.0
    for layer in model.children():
        if isinstance(layer, nn.Linear):  # Apply to Linear layers
            neuron_weights = layer.weight  # Shape: [out_features, in_features]
            neuron_biases = layer.bias     # Shape: [out_features]
            
            # Apply norm penalty to each neuron
            for i in range(layer.out_features):
                weight_penalty = l1_loss_zero_one(neuron_weights[i])
                bias_penalty = l1_loss_zero_one(neuron_biases[i])
                penalty += weight_penalty + bias_penalty
    return penalty

def run(n_tasks,
        n,
        k,
        D,
        width,
        depth,
        activation,
        test_points,
        test_points_per_task,
        steps,
        epochs,
        batch_size,
        lr,
        weight_decay,
        device,
        dtype,
        log_freq,
        verbose,
        seed):

    torch.set_default_dtype(dtype)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    info = {}
    models_saved = []

    mlp = MLP(activation, depth, width).to(device)
    ### create model ###
    p = 59
    d = 32
    in_dim = 2*d
    out_dim = p

    shp = [n + n_tasks, width, 2]
    #mlp = BioMLP(shp=shp, token_embedding=True, embedding_size=(2, (n + n_tasks) // 2)).to(device)
    #mlp = BioMLP(shp=shp, token_embedding=False).to(device)
    info['P'] = sum(t.numel() for t in mlp.parameters())

    Ss = get_subsets(n_tasks, n, k)
    info['Ss'] = Ss

    probs = np.array([np.power(n, -alpha) for n in range(1+offset, n_tasks+offset+1)])
    probs = probs / np.sum(probs)
    cdf = np.cumsum(probs)

    test_batch_sizes = [int(prob * test_points) for prob in probs]

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.98))
    lamb = 0.1
    swap_log = log_freq
    
    info['accuracies'] = list()
    info['losses'] = list()
    info['losses_subtasks'] = dict()
    info['accuracies_subtasks'] = dict()
    for i in range(n_tasks):
        info['losses_subtasks'][str(i)] = list()
        info['accuracies_subtasks'][str(i)] = list()

    train_loader, train_loaders_subtasks = get_data(steps, batch_size, cdf, n_tasks, n, Ss, device, dtype, test_points_per_task)

    initial_weight = 1e-10
    final_weight = 1e-0
        
    for epoch in tqdm(range(epochs), disable=not verbose):
        if epoch == int(epochs*1/4):
            lamb *= 10
        
        if epoch == int(epochs*3/4):
            lamb *= 0.1
            
        current_weight = initial_weight + (final_weight - initial_weight) * (epoch / epochs)
    
        train_loss, train_accuracy = train_one_epoch(mlp, train_loader, train_loaders_subtasks, optimizer, loss_fn, device, n_tasks, current_weight)
        if epoch % log_freq == 0:
            info['accuracies'].append(train_accuracy) 
            info['losses'].append(train_loss)
            for task in range(n_tasks):
                loss, accuracy = evaluate(mlp, train_loaders_subtasks[task], loss_fn, device)
                info['losses_subtasks'][str(task)].append(loss)
                info['accuracies_subtasks'][str(task)].append(accuracy)
            models_saved += [copy.deepcopy(mlp)]
        #if epoch % swap_log == 0 and epoch > 0:
            #mlp.relocate()
        
    return info, models_saved, loss_fn, train_loader, train_loaders_subtasks, Ss

n_tasks = 5
n = 50
k = 3
alpha = 1.5
offset = 0

D = -1 # -1 for infinite data

width = 3000
depth = 2
activation = 'ReLU'
    
steps = 10
batch_size = 30000
lr = 1e-4
weight_decay = 0.0
test_points = 30000
test_points_per_task = 1000
epochs = 3000
stop_early = False
    
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

log_freq = epochs // 100
verbose=True
seed = 0
runs = 1


info, models_saved, criterion, train_loader, train_loaders_subtasks, Ss = run(n_tasks,
    n, 
    k, 
    D, 
    width, 
    depth, 
    activation, 
    test_points, 
    test_points_per_task, 
    steps,
    epochs,
    batch_size, 
    lr, 
    weight_decay, 
    device, 
    dtype, 
    log_freq, 
    verbose, 
    seed)


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

In [ ]:
criterion = nn.CrossEntropyLoss()
Ss = get_subsets(n_tasks, n, k)
probs = np.array([np.power(n, -alpha) for n in range(1+offset, n_tasks+offset+1)])
probs = probs / np.sum(probs)
cdf = np.cumsum(probs)
test_batch_sizes = [int(prob * test_points) for prob in probs]

In [ ]:
from devinterp.slt.sampler import estimate_learning_coeff_with_summary
from devinterp.utils import evaluate_ce

N_EPOCHS = epochs
SAVE_EVERY_N_EPOCHS = log_freq

def estimate_rlcts(models, param_dict, train_loader, criterion, data_length, device, num_draws, num_models):
    estimates = {"sgnht": [], "sgld": []}
    losses = None
    for idx, model in enumerate(tqdm(models)):
        for method, optimizer_kwargs in [
            ("sgld", {"lr": 1e-5, "localization": 100.0, "noise_level": 1.0}),
        ]:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                evaluate=evaluate_ce,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=1,
                num_draws=num_draws,
                num_burnin_steps=100,
                num_steps_bw_draws=1,
                device=device,
                seed=0,
                optimize_over_per_model_param=param_dict
            )
            estimate = results["llc/mean"]

            # take losses from last chain for plotting
            if idx == (N_EPOCHS // SAVE_EVERY_N_EPOCHS) - 1:
                losses = results['loss/trace']
            estimates[method].append(estimate)
    return estimates, losses

def obtain_rlct_estimates(train_loader, models_saved, param_dict, criterion, runs):
    num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS
    data_length = len(train_loader)
    rlct_estimates = {"sgnht": torch.zeros(runs, num_models), "sgld": torch.zeros(runs, num_models)}
    num_draws = 400
    last_chain_losses = torch.zeros(runs, num_draws)

    for run in tqdm(range(runs)):
        rlct_estimate, losses = estimate_rlcts(
            models_saved[num_models * run : num_models * (run + 1)], param_dict, train_loader, criterion, data_length, device, num_draws, num_models
        )
        rlct_estimates["sgld"][run] = torch.tensor(rlct_estimate["sgld"])
        last_chain_losses[run] = torch.tensor(losses)

    rlct_estimates_final = {"sgnht": rlct_estimates["sgnht"].mean(dim=0), "sgld": rlct_estimates["sgld"].mean(dim=0)}
    return rlct_estimates_final, last_chain_losses.mean(dim=0)

#rlct_estimates_final, last_chain_losses_final = obtain_rlct_estimates(train_loader, models_saved, None, criterion, runs)

In [ ]:
'''
from devinterp.slt.sampler import  sample, LLCEstimator
from devinterp.optim import SGLD
from devinterp.utils import default_nbeta

# Assuming you have a PyTorch Model assigned to model, and DataLoader assigned to trainloader
llc_estimator = LLCEstimator(..., nbeta=default_nbeta(train_loader))
sample(models_saved[-1], train_loader, ..., callbacks = [llc_estimator])

llc_mean = llc_estimator.get_results()["llc/mean"]
print(llc_mean)
'''

In [ ]:
dataset = '0'

def plot_losses(train_losses_final, name = ''):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_losses_final, label="Train Loss, sgd", color=PRIMARY)
    #ax1.plot(x_axis, test_losses_final, label="Test Loss, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_subtask_losses(train_losses_subtasks, n_tasks, name='full'):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    
    for task in range(n_tasks):
        ax1.plot(x_axis, train_losses_subtasks[str(task)], label="Task " + str(task), color=sns.color_palette("muted")[task])
        
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_subtasks_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_accuracies(train_accuracies_final, name = ''):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_accuracies_final, label="Train Accuracy, sgd", color=PRIMARY)
    #ax1.plot(x_axis, test_accuracies_final, label="Test Accuracy, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_subtask_accuracies(train_accuracies_subtasks, n_tasks, name='full'):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    
    for task in range(n_tasks):
        ax1.plot(x_axis, train_accuracies_subtasks[str(task)], label="Task " + str(task), color=sns.color_palette("muted")[task])
        
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_subtasks_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_rlcts(rlct_estimates_final, dataset, rlct_estimates_final_other = {}):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    if rlct_estimates_final_other != {}:
        ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="Average of ablation curves", color=QUATERNARY)
    ax2.plot(x_axis, rlct_estimates_final["sgld"], label="Original", color=TERTIARY_LIGHT)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_" + dataset + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_rlcts_multiple_curves(rlct_curves, rlct_selector, rlct_original, dataset):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS + 1, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)

    ax1.plot(x_axis, rlct_original['sgld'], label='Full model', color=sns.color_palette("muted")[n_tasks + 1])
    ax1.plot(x_axis, rlct_selector['sgld'], label='Selector', color=sns.color_palette("muted")[n_tasks + 2])
    
    for task in range(n_tasks):
        ax1.plot(x_axis, rlct_curves[task]['sgld'], label="Task " + str(task), color=sns.color_palette("muted")[task])
        
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig(dataset + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_losses_chain(last_chain_losses_final, dataset):
    sns.set_style("whitegrid")
    

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Draw")
    ax1.set_ylabel("Loss", color=PRIMARY)
    ax1.plot(last_chain_losses_final, label="Loss, sgd", color=PRIMARY)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("last_chain_losses_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

train_losses_final = info['losses']
train_accuracies_final = info['accuracies']
    
plot_losses(train_losses_final, dataset)
plot_subtask_losses(info['losses_subtasks'], n_tasks)
plot_accuracies(train_accuracies_final, dataset)
plot_subtask_accuracies(info['accuracies_subtasks'], n_tasks)
#plot_rlcts(rlct_estimates_final, dataset='full')
#plot_losses_chain(last_chain_losses_final, dataset)

In [ ]:
def return_topk_percent_mask(tensor, proportion):
    # Step 1: Flatten the tensor
    flattened_tensor = tensor.flatten()

    # Step 2: Determine K, where K is 20% of the total number of elements
    total_elements = flattened_tensor.numel()
    K = int(proportion * total_elements)

    # Step 3: Find the value of the K-th largest element
    if K > 0:
        topk_values, _ = torch.topk(flattened_tensor, K)
        threshold_value = topk_values[-1]
    else:
        return tensor >= 0

    # Step 4: Create a boolean mask of the top K values
    return tensor >= threshold_value


def unravel_index(index, shape):
    out = []
    for dim in reversed(shape):
        out.append(index % dim)
        index = index // dim
    return tuple(reversed(out))

def ablation_study(model, loss_fn):
    
    loss_diffs_per_task = []
    for index in tqdm(range(n_tasks)):
        loss_diffs = {}
        for name, param in model.named_parameters():
            loss_diffs[name] = torch.zeros(param.shape)
            x_i, y_i = get_batch(n_tasks=n_tasks, n=n, Ss=[Ss[index]], codes=[index], sizes=[test_points_per_task], device=device, dtype=dtype)
            y_i_pred = model(x_i)
            loss_baseline = loss_fn(y_i_pred, y_i).item()
            
            for idx in tqdm(range(param.numel())):
                with torch.no_grad():
                    # Convert flat index i to multi-dimensional index for the original shape
                    multi_idx = unravel_index(idx, param.shape)
                
                    # Save the original weight value
                    original_value = param[multi_idx].item()
                
                    # Set the weight to zero
                    param[multi_idx] = 0.0
                    
                    y_i_ablated = model(x_i)
                    loss_ablated = loss_fn(y_i_ablated, y_i).item()
                    
                    loss_diffs[name][multi_idx] = abs(loss_ablated - loss_baseline)
                    
                    # Restore the original weight
                    param[multi_idx] = original_value
                    torch.cuda.empty_cache()
        loss_diffs_per_task.append(loss_diffs)
                    
    return loss_diffs_per_task   

def neuron_ablation_study(model, loss_fn):
    loss_diffs_per_task = []
    
    for index in tqdm(range(n_tasks)):
        loss_diffs = {}
        
        # Get baseline loss
        x_i, y_i = get_batch(n_tasks=n_tasks, n=n, Ss=[Ss[index]], codes=[index], 
                            sizes=[test_points_per_task], device=device, dtype=dtype)
        y_i_pred = model(x_i)
        loss_baseline = loss_fn(y_i_pred, y_i).item()
        
        # For each layer in the model
        for i, layer in enumerate(model.model):
            print(i, layer)
            # Only process Linear layers (skip activation layers)
            if isinstance(layer, nn.Linear):
                loss_diffs[f'layer_{i}'] = []
                
                # For each neuron in the layer
                for neuron_idx in range(layer.out_features):
                    with torch.no_grad():
                        # Store original values
                        original_weights = layer.weight[neuron_idx].clone()
                        original_bias = layer.bias[neuron_idx].clone()
                        
                        # Zero out the entire neuron
                        layer.weight[neuron_idx].zero_()
                        layer.bias[neuron_idx] = 0.0
                        
                        # Compute new loss
                        y_i_ablated = model(x_i)
                        loss_ablated = loss_fn(y_i_ablated, y_i).item()
                        
                        # Store difference
                        loss_diff = abs(loss_ablated - loss_baseline)
                        loss_diffs[f'layer_{i}'].append(loss_diff)
                        
                        # Restore original values
                        layer.weight[neuron_idx].copy_(original_weights)
                        layer.bias[neuron_idx].copy_(original_bias)
                        
                        torch.cuda.empty_cache()
                
                # Convert list to tensor
                loss_diffs[f'layer_{i}'] = torch.tensor(loss_diffs[f'layer_{i}'])
                
        loss_diffs_per_task.append(loss_diffs)
                    
    return loss_diffs_per_task

# Use the function
#loss_diffs_per_task = ablation_study(models_saved[-1], criterion)
loss_diffs_per_task = neuron_ablation_study(models_saved[-1], criterion)

In [ ]:
print(loss_diffs_per_task[0])

for key, value in loss_diffs_per_task[0].items():
    print(key, value.shape)

In [ ]:
indices_per_task = []
use_variable_thresholds = False

# Analyze results
for task in tqdm(range(n_tasks)):
    
    indices_dict = {}

    for key in loss_diffs_per_task[task].keys():
        if use_variable_thresholds:
            indices_dict[key] = loss_diffs_per_task[task][key] >= thresholds_per_task[task][key]
        else:
            proportion = .05
            indices_dict[key] = return_topk_percent_mask(loss_diffs_per_task[task][key], proportion)
            '''
            if 'weight' in key:
                indices_dict[key] = return_topk_percent_mask(loss_diffs_per_task[task][key], proportion)
            else:
                indices_dict[key] = torch.ones(loss_diffs_per_task[task][key].shape, dtype=torch.bool)
            '''
  
    indices_per_task.append(indices_dict)

In [ ]:
for task in range(n_tasks):
    for key, value in indices_per_task[task].items():
        for task1 in range(n_tasks):
            for key1, val1 in indices_per_task[task1].items():
                if value.shape == val1.shape:
                    # Element-wise equality
                    overlap = value == val1

                    # Count the number of matches
                    num_matches = torch.sum(overlap).item()

                    # Total number of elements
                    total_elements = value.numel()

                    # Proportion of overlap
                    proportion_overlap = num_matches / total_elements
                    print(key, key1, torch.all(value == val1), proportion_overlap)

In [ ]:
loss_diffs_averaged_across_tasks = {}

for key in loss_diffs_per_task[0].keys():
    mean_tensor = sum(d[key] for d in loss_diffs_per_task) / n_tasks
    sorted_flattened_tensor, _ = mean_tensor.flatten().sort(descending=True)
    loss_diffs_averaged_across_tasks[key] = mean_tensor
    sns.set_style("whitegrid")
    x_axis = np.arange(0, len(sorted_flattened_tensor), 1)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("neuron")
    ax1.set_ylabel("mean loss diff", color=PRIMARY)
    ax1.plot(x_axis, sorted_flattened_tensor, label="mean loss diff", color=PRIMARY)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("cum_loss_diff_in_layer_" + key + ".png")

# set threshold manually on basis of plots
thresholds = {'layer_0': 0.015, 
              'layer_2': 0.0}
'''
thresholds = {'model.0.weight': 0.0005, 
              'model.0.bias': 0.001, 
             'model.2.weight': 0.0, 
             'model.2.bias': 0.0}
             '''

selector_circuit_indices = {}

for key in loss_diffs_per_task[0].keys():
    selector_circuit_indices[key] = loss_diffs_averaged_across_tasks[key] > thresholds[key]
    
#for task in tqdm(range(n_tasks)):
#    for key in loss_diffs_per_task[0].keys():
#        indices_per_task[task][key] &= ~selector_circuit_indices[key]
        #for other_task in range(n_tasks):
            #if task == other_task:
            #    continue
            #indices_per_task[task][key] &= indices_per_task[other_task][key]
        


In [ ]:
def get_path_losses_per_task(loss_diffs_per_task, n_tasks):
    path_losses_per_task = []
    path_indices_per_task = []
    indices_per_task = []

    for task in tqdm(range(n_tasks)):
        W_0 = loss_diffs_per_task[task]['model.0.weight']
        b_0 = loss_diffs_per_task[task]['model.0.bias']
        W_1 = loss_diffs_per_task[task]['model.2.weight']
        b_1 = loss_diffs_per_task[task]['model.2.bias']
        
        indices_dict = {}
        
        indices_per_task.append(indices_dict)
    
        path_losses = W_0[ : , : , None, None] + W_1[None, None, : , : ]
        epsilon = 0.04
        indices =  path_losses > epsilon
        indices_dict['model.0.weight'] = indices.any(dim=(2, 3))
        indices_dict['model.0.bias'] = torch.ones(b_0.shape, dtype=torch.bool)
        indices_dict['model.2.weight'] = indices.any(dim=(0, 1))
        indices_dict['model.2.bias'] = torch.ones(b_1.shape, dtype=torch.bool)

        print(indices_dict['model.0.weight'])
        print(indices_dict['model.0.bias'])
        print(indices_dict['model.2.weight'])
        print(indices_dict['model.2.bias'])
        indices_per_task.append(indices_dict)
        path_losses_per_task.append(path_losses.flatten())
        
    return path_losses_per_task, indices_per_task

for name, param in models_saved[-1].named_parameters():
    print(name)
    print(param.shape)

#path_losses_per_task, indices_per_task = get_path_losses_per_task(loss_diffs_per_task, n_tasks)

In [ ]:
print(indices_per_task[0])

In [ ]:
import copy

# Check if models have the same parameters
def check_models_equal(model1, model2):
    model1_params = model1.state_dict()
    model2_params = model2.state_dict()

    # Ensure both models have the same keys
    if model1_params.keys() != model2_params.keys():
        return False

    # Compare each parameter tensor
    for key in model1_params.keys():
        if not torch.equal(model1_params[key], model2_params[key]):
            print(f"Mismatch found in parameter: {key}")
            return False

    return True

def prune_to_obtain_circuit_other(model, model_indices, selector_indices, compliment=False):

    for idx, layer in enumerate(model.model):
        if isinstance(layer, nn.Linear):
            for neuron_idx in range(layer.out_features):
                if selector_indices == None:
                    if not compliment:
                        if not model_indices[f'layer_{idx}'][neuron_idx]:  # if False, zero out this neuron
                            with torch.no_grad():
                                layer.weight[neuron_idx].zero_() # zero all weights connected to this neuron
                                layer.bias[neuron_idx] = 0.0    # zero the bias for this neuron
                    else:
                        if model_indices[f'layer_{idx}'][neuron_idx]:  # if False, zero out this neuron
                            with torch.no_grad():
                                layer.weight[neuron_idx].zero_() # zero all weights connected to this neuron
                                layer.bias[neuron_idx] = 0.0    # zero the bias for this neuron
                    
                else:
                    if not compliment:
                        if not model_indices[f'layer_{idx}'][neuron_idx] and not selector_indices[f'layer_{idx}'][neuron_idx]:  # if False, zero out this neuron
                            with torch.no_grad():
                                layer.weight[neuron_idx].zero_() # zero all weights connected to this neuron
                                layer.bias[neuron_idx] = 0.0    # zero the bias for this neuron
                    else:
                        if model_indices[f'layer_{idx}'][neuron_idx] or selector_indices[f'layer_{idx}'][neuron_idx]:  # if False, zero out this neuron
                            with torch.no_grad():
                                layer.weight[neuron_idx].zero_() # zero all weights connected to this neuron
                                layer.bias[neuron_idx] = 0.0    # zero the bias for this neuron

    return model

def prune_to_obtain_circuit(model, model_indices, selector_indices, compliment=False):
    
    model_state_dict = model.state_dict()
    
    for name, param in model.named_parameters():
        indices = model_indices[name]
        if selector_indices == None:
            if not compliment:
                model_state_dict[name][~indices] = 0.0
            else:
                model_state_dict[name][indices] = 0.0
        else:
            if not compliment:
                model_state_dict[name][~(indices | selector_indices[name])] = 0.0
            else:
                model_state_dict[name][indices | selector_indices[name]] = 0.0
        
    model.load_state_dict(model_state_dict)
            
    return model

def obtain_list_of_models_per_circuit(models_saved, indices_per_task, selector_indices, n_tasks):

    models_per_task = []
    model_compliments_per_task = []

    for task in tqdm(range(n_tasks)):
        models = []
        model_compliments = []
        for model in tqdm(models_saved):
            task_model = copy.deepcopy(model)
            task_model = prune_to_obtain_circuit_other(task_model, indices_per_task[task], selector_indices)
            models.append(task_model)
            task_model_compliment = copy.deepcopy(model)
            task_model_compliment = prune_to_obtain_circuit_other(task_model_compliment, indices_per_task[task], selector_indices, True)
            model_compliments.append(task_model_compliment)
        models_per_task.append(models)
        model_compliments_per_task.append(model_compliments)

    for num1, models1 in enumerate(models_per_task):
        for num2, models2 in enumerate(models_per_task):
            if num1 == num2:
                continue
            num_equal = 0
            for (model1, model2) in zip(models1, models2):
                num_equal += int(check_models_equal(model1, model2))
            print(num_equal, len(models1))
        
    return models_per_task, model_compliments_per_task

models_per_task_with_selector, model_compliments_per_task_with_selector = obtain_list_of_models_per_circuit(models_saved, indices_per_task, selector_circuit_indices, n_tasks)
models_per_task_without_selector, model_compliments_per_task_without_selector = obtain_list_of_models_per_circuit(models_saved, indices_per_task, None, n_tasks) 

In [ ]:
def compute_loss_curve_for_model(models, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype):
    losses = []
    accuracies = []
    losses_subtasks = {}
    accuracies_subtasks = {}
    loss_fn = nn.CrossEntropyLoss()
    
    for i in range(n_tasks):
        losses_subtasks[str(i)] = list()
        accuracies_subtasks[str(i)] = list()

    for index, model in tqdm(enumerate(models)):
        train_loss = 0
        train_accuracy = 0
        coeff = 1.0
        for index, (data, targets) in enumerate(train_loader):
            outputs = model(data.to(device))
            loss = criterion(outputs, targets.to(device))
            train_loss += loss.item()
            train_accuracy += accuracy_function(outputs, targets.to(device)).item()
        for task in range(n_tasks):
            subtask_losses = 0
            subtask_accuracies = 0
            for (data_subtask, targets_subtask) in train_loaders_subtasks[task]:
                outputs_subtask = model(data_subtask.to(device))
                loss_subtask = criterion(outputs_subtask, targets_subtask.to(device))
                subtask_losses += loss_subtask.item()
                accuracy_subtask = accuracy_function(outputs_subtask, targets_subtask.to(device))
                subtask_accuracies += accuracy_subtask.item()
            losses_subtasks[str(task)].append(subtask_losses / len(train_loaders_subtasks[task]))
            accuracies_subtasks[str(task)].append(subtask_accuracies / len(train_loaders_subtasks[task]))
                    
        accuracies.append(train_accuracy / len(train_loader)) 
        losses.append(train_loss / len(train_loader))
    return losses, accuracies, losses_subtasks, accuracies_subtasks

def plot_loss_curves_for_circuits(models_per_task, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype, compliment='', selector=''):
    for task in range(n_tasks):
        losses, accuracies, losses_subtasks, accuracies_subtasks = compute_loss_curve_for_model(models_per_task[task], train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype)
    
        plot_losses(losses, 'task_'+ str(task) + '_with_ablation' + compliment + selector)
        plot_accuracies(accuracies, 'task_'+ str(task) + '_with_ablation' + compliment + selector)
        plot_subtask_losses(losses_subtasks, n_tasks, 'task_'+ str(task) + '_with_ablation' + compliment + selector)
        plot_subtask_accuracies(accuracies_subtasks, n_tasks, 'task_'+ str(task) + '_with_ablation' + compliment + selector)
        
plot_loss_curves_for_circuits(models_per_task_with_selector, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype)
plot_loss_curves_for_circuits(model_compliments_per_task_with_selector, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype, '_compliment')
plot_loss_curves_for_circuits(models_per_task_without_selector, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype, '', '_without_selector')
plot_loss_curves_for_circuits(model_compliments_per_task_without_selector, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype, '_compliment', '_without_selector')

In [27]:
def obtain_pruned_full_model(models_saved, indices_for_all_tasks, selector_indices):
    models = []
    
    for model in tqdm(models_saved):
        task_model = copy.deepcopy(model)
        task_model = prune_to_obtain_circuit(task_model, indices_for_all_tasks, selector_indices)
        models.append(task_model)
    
    return models

def get_indices_for_all_tasks(indices_per_task):
    indices_for_all_tasks = {}

    for key in indices_per_task[0].keys():
        indices_for_all_tasks[key] = indices_per_task[0][key]
        
        for task in range(1, n_tasks):
            indices_for_all_tasks[key] |= indices_per_task[task][key]
    
    return indices_for_all_tasks
        
indices_for_all_tasks = get_indices_for_all_tasks(indices_per_task)
models_full_pruned = obtain_pruned_full_model(models_saved, indices_for_all_tasks, selector_circuit_indices)

  0%|          | 0/100 [00:00<?, ?it/s]

KeyError: 'model.0.weight'

In [ ]:
losses, accuracies, losses_subtasks, accuracies_subtasks = compute_loss_curve_for_model(models_full_pruned, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype)
    
plot_losses(losses, 'full_model_with_ablation')
plot_accuracies(accuracies, 'full_model_with_ablation')
plot_subtask_losses(losses_subtasks, n_tasks, 'full_model_with_ablation')
plot_subtask_accuracies(accuracies_subtasks, n_tasks, 'full_model_with_ablation')

In [ ]:
rlcts_per_task = []
last_chain_losses_per_task = []


for task in tqdm(range(n_tasks)):
    rlct_estimates, last_chain_losses = obtain_rlct_estimates(train_loader, models_saved, indices_per_task[task], criterion, runs)
    rlcts_per_task.append(rlct_estimates)
    last_chain_losses_per_task.append(last_chain_losses)

rlct_estimates_selector, _ = obtain_rlct_estimates(train_loader, models_saved, selector_circuit_indices, criterion, runs)

In [ ]:
for task in range(n_tasks):
    plot_rlcts(rlcts_per_task[task], dataset='ablated_model_for_task_' + str(task))
    plot_losses_chain(last_chain_losses_per_task[task], dataset='ablated_model_for_task_' + str(task))
plot_rlcts(rlct_estimates_selector, dataset='ablated_model_for_selector')

In [ ]:
total_rlcts = {}
total_rlcts['sgld'] = rlcts_per_task[0]['sgld']

for task in range(1, n_tasks):
    total_rlcts['sgld'] += rlcts_per_task[task]['sgld']

total_rlcts['sgld'] /= n_tasks

plot_rlcts(rlct_estimates_final, dataset='rlct_curve_for_summed_curves vs. original', rlct_estimates_final_other=total_rlcts)

In [ ]:
plot_rlcts_multiple_curves(rlcts_per_task, rlct_estimates_selector, rlct_estimates_final, 'rlct_ablated_model_curves_vs_full_model')

In [ ]:
def compute_overlap(A, B):
    intersection = torch.logical_and(A, B)
    union = torch.logical_or(A, B)

    # Compute degree of overlap (Jaccard index)
    overlap = intersection.sum().float() / union.sum().float()
    return overlap

for layer in indices_per_task[0].keys():
    for task1 in range(n_tasks):
        A = indices_per_task[task1][layer]
        for task2 in range(task1 + 1, n_tasks):
            B = indices_per_task[task2][layer]
            print(f"{layer}: similarity between {task1} and {task2} is {compute_overlap(A, B)}")
    

In [ ]:
model = 'mlp'
torch.save(models_saved, 'models_saved_' + str(N_EPOCHS) + '_' + model + '.pt')
torch.save(info, 'info_' + str(N_EPOCHS) + '_' + model + '.pt')
torch.save(train_loader, 'train_loader_' + str(N_EPOCHS) + '_' + model + '.pt')
torch.save(train_loaders_subtasks, 'train_loaders_subtasks_' + str(N_EPOCHS) + '_' + model + '.pt')
torch.save(rlct_estimates_final, 'rlct_estimates_full_' + str(N_EPOCHS) + '_' + model + '.pt')
torch.save(loss_diffs_per_task, 'loss_diffs_per_task_' + str(N_EPOCHS) + '_' + model + '.pt')

In [ ]:
for task in range(n_tasks):
    torch.save(rlcts_per_task[task], 'rlct_estimates_task_' + str(task) + '_' + str(N_EPOCHS) + '_' + model + '.pt')

torch.save(rlct_estimates_selector, 'rlct_estimates_selector_' + str(N_EPOCHS) + '_' + model + '.pt')